Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso para desbloquear el cuaderno.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Modelos de adelantamiento

En este cuaderno construiremos un **modelo de adelantamiento** a partir de los Grandes Premios de México de 2023, 2024 y 2025.

Usaremos una **regresión logística** para obtener las probabilidades de adelantamiento en función de la **diferencia de ritmo teórico** entre los dos coches, calculada a partir de su ritmo de base y de sus neumáticos.

Los datos vienen de [OpenF1](https://openf1.org), una API pública y gratuita.

## Inicialización

### Importar librerías

Importamos las librerías relevantes.

In [ ]:
# Librerías propias (ver src/race_analysis y src/race_models)
from race_analysis.ingestion.openf1 import load_race, get
from race_analysis.ingestion.laps import clean_data
from race_analysis.tyre_deg.pace import fit_pooled_race_pace
from race_analysis.tyre_deg.degradation_plots import plot_track_evolution
from race_analysis.overtaking_model.overtaking import find_overtakes, attack_situations, fit_logistic
from race_analysis.overtaking_model.overtaking_plots import plot_logistic
from race_models.f1.models import base_pace_from_race_pace
from race_models.f1.analysis.race_plots import plot_base_pace, plot_tyre_model

# Librerías de Python
import pandas as pd

### Descarga de datos

`load_race` descarga las vueltas, stints, compuestos y otros datos relevantes de la API.

In [ ]:
TRACK = "Mexico"
YEARS = [2023, 2024, 2025]

races = {year: load_race(year, TRACK) for year in YEARS}

### Limpiar los datos

Limpiamos los datos descartando la primera vuelta de la carrera, vueltas bajo neutralización y vueltas lentas, entre otros.

In [ ]:
laps = {year: clean_data(race) for year, race in races.items()}

### Efecto del combustible y evolución de la pista

Corregimos el efecto del combustible y la evolución de la pista para poder fitear correctamente un modelo de degradación.

In [ ]:
FUEL_EFFECT = 0.05  # s por vuelta de combustible a bordo (estimación habitual en F1)
for frame in laps.values():
    frame["fuel_corrected"] = frame["lap_time"] - FUEL_EFFECT * frame["laps_remaining"]

In [ ]:
# La evolución de la pista la estima el modelo de ritmo, con las tres carreras juntas.
paces = fit_pooled_race_pace(pd.concat(laps.values(), ignore_index=True))

## 2. Generar un modelo de neumáticos

Generamos un modelo de neumáticos lineal (ver el cuaderno de la clase 4.3) para poder cuantificar los ritmos teóricos en cada momento.

In [ ]:
tlas = {driver["driver_number"]: driver["name_acronym"] for driver in get("drivers", session_key=races[2025].session_key)}

linear = fit_pooled_race_pace(pd.concat(laps.values(), ignore_index=True), model="lineal")

plot_tyre_model(linear[2025].tyre_curves(), max_life=45, title="México 2023-2025 · Modelo lineal");

plot_base_pace(base_pace_from_race_pace(linear[2025], tlas), title="México 2025 · Ritmo base");

## 3. Situaciones de ataque y adelantamientos

Para calcular una probabilidad no basta con contar los adelantamientos: también necesitamos la cantidad de **adelantamientos intentados**.

Definimos las **situaciones de ataque** de esta manera: una vuelta que un coche empieza a menos de `MAX_GAP` segundos del coche que tiene justo delante, en bandera verde y sin que ninguno de los dos entre o salga de boxes. Si el coche de detrás adelanta a ese coche durante la vuelta, el ataque tiene éxito.

In [ ]:
MAX_GAP = 1.5  # distancia máxima (s) con el coche de delante para considerar que hay un ataque

Encontramos los adelantamientos que vienen de situaciones de ataque. `OpenF1` registra cada cambio de posición, pero la mayoría no son adelantamientos que nos sirvan para el modelo, así que `find_overtakes` descarta los cambios de posición en la salida, en boxes, bajo neutralización, por incidentes y los que no vienen de una situación de ataque, y se queda solo con los **válidos**.

In [ ]:
# Adelantamientos válidos: los que vienen de una situación de ataque
overtaking_data = {year: find_overtakes(race, laps[year], max_gap=MAX_GAP) for year, race in races.items()}

for year in YEARS:
    print(f"{year}: {len(overtaking_data[year].registered)} adelantamientos registrados por OpenF1, "
          f"{len(overtaking_data[year].overtakes)} válidos")

# Situaciones de ataque, con la diferencia de ritmo teórico y si acaban en adelantamiento
attacks = {year: attack_situations(races[year], laps[year], linear[year], overtaking_data[year], max_gap=MAX_GAP)
           for year in YEARS}
all_attacks = pd.concat(attacks.values(), ignore_index=True)

print(f"Situaciones de ataque: {len(all_attacks)}")
print(f"Adelantamientos: {all_attacks['overtake'].sum()} ({all_attacks['overtake'].mean():.0%})")

In [ ]:
# 5 situaciones de ataque de 2025 al azar
sample = attacks[2025].sample(5, random_state=42).sort_values("lap_number")
pd.DataFrame({
    "Vuelta": sample["lap_number"],
    "Atacante": sample["attacker"].map(tlas),
    "Defensor": sample["defender"].map(tlas),
    "Distancia (s)": sample["gap"].round(2),
    "Neumático atacante": sample["attacker_compound"].str[0] + sample["attacker_tyre_life"].astype(str),
    "Neumático defensor": sample["defender_compound"].str[0] + sample["defender_tyre_life"].astype(str),
    "Δ ritmo (s/vuelta)": sample["delta_pace"].round(2),
    "Adelanta": sample["overtake"].map({True: "Sí", False: "No"}),
}).reset_index(drop=True)

## 4. Regresión logística

Obtenemos las probabilidades de adelantar en función de la diferencia de ritmo teórico utilizando una **regresión logística**.

In [ ]:
overtaking_model = fit_logistic(all_attacks["delta_pace"], all_attacks["overtake"])

plot_logistic(all_attacks, "delta_pace", overtaking_model, bins=[-3, -1, -0.5, 0, 0.5, 1, 1.5, 2, 4],
              xlabel="Diferencia de ritmo teórico (s/vuelta, positiva si el atacante es más rápido)",
              title="México 2023-2025 · Diferencia de ritmo teórico");